In [0]:
# -----------------------------
# Paths
# -----------------------------

source_path = "/Volumes/workspace/default/my_volume_1/orders_raw.csv"

bronze_path = "/Volumes/workspace/default/my_volume_1/bronze_orders"
silver_path = "/Volumes/workspace/default/my_volume_1/silver_orders"

gold_city_path = "/Volumes/workspace/default/my_volume_1/gold_sales_by_city"
gold_category_path = "/Volumes/workspace/default/my_volume_1/gold_sales_by_category"
gold_product_path = "/Volumes/workspace/default/my_volume_1/gold_top_products"

# -----------------------------
# BRONZE LAYER
# Raw ingestion
# -----------------------------

bronze_df = (
    spark.read.format("csv")
    .option("header","true")
    .option("inferSchema","true")
    .load(source_path)
    .withColumn("ingestion_time", current_timestamp())
)

bronze_df.write.format("delta") \
.mode("overwrite") \
.save(bronze_path)

print("Bronze layer created")

# -----------------------------
# SILVER LAYER
# Data cleaning
# -----------------------------

bronze_df = spark.read.format("delta").load(bronze_path)

silver_df = (
    bronze_df
    .dropDuplicates()
    .filter(col("quantity").isNotNull())
    .filter(col("price").isNotNull())
    .withColumn("order_date", to_date("order_date"))
    .withColumn("total_amount", col("quantity") * col("price"))
)

silver_df.write.format("delta") \
.mode("overwrite") \
.save(silver_path)

print("Silver layer created")

# -----------------------------
# GOLD LAYER
# Business aggregations
# -----------------------------

silver_df = spark.read.format("delta").load(silver_path)

# 1 Sales by city

gold_city_df = silver_df.groupBy("city") \
    .agg(sum("total_amount").alias("total_sales"))

gold_city_df.write.format("delta") \
.mode("overwrite") \
.save(gold_city_path)

# 2 Sales by category

gold_category_df = silver_df.groupBy("category") \
    .agg(sum("total_amount").alias("total_revenue"))

gold_category_df.write.format("delta") \
.mode("overwrite") \
.save(gold_category_path)

# 3 Top selling products

gold_product_df = silver_df.groupBy("product") \
    .agg(count("order_id").alias("total_orders"))

gold_product_df.write.format("delta") \
.mode("overwrite") \
.save(gold_product_path)

print("Gold layer created")

# -----------------------------
# Display results
# -----------------------------

print("Sales by City")
display(gold_city_df)

print("Sales by Category")
display(gold_category_df)

print("Top Products")
display(gold_product_df)